# Day 2, Notebook 2: cross the file boundary

Restart the kernel and your records are gone. That is not a bug in your code. It is what memory means.

This notebook takes the functions you carved in notebook 1 and points them at real files, in both directions.

Position in the day:

`[inline cell] > [packaged decision] > [read the failure] > [log the rejection] > **[cross the boundary]**`

## Setup

Everything this notebook needs, in one cell at the top, so it runs cold in a fresh Codespace.

In [ ]:
import csv
import json
import os
import traceback

# csv and json are today's topic. os and traceback are plumbing: one makes a folder,
# the other lets a failure print itself without stopping the notebook. Not topics today.

RECORDS_CSV = "C2_W01_D02_data_records_STUDENT.csv"
RECORDS_JSON = "C2_W01_D02_data_records_STUDENT.json"
TRUNCATED_JSON = "C2_W01_D02_data_vendor_truncated_STUDENT.json"
OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def show_failure(fn):
    """Run something that is meant to fail and print its real traceback."""
    try:
        fn()
    except Exception:
        print(traceback.format_exc())

print("setup done, writing outputs into", OUTPUT_DIR)

## Section 1: the wrong path, on purpose

Before opening a file correctly, look at what a wrong path says. This is the cheapest error in the whole programme.

In [ ]:
def open_a_wrong_path():
    return open("data/orderz.csv")

show_failure(open_a_wrong_path)

```
FileNotFoundError: [Errno 2] No such file or directory: 'data/orderz.csv'
```

It names the exact path it tried. Read that path out loud and you find the typo yourself, without a search engine.

Two things worth knowing before you edit anything:

1. The path is relative to where the kernel is running, not to where the file browser is pointing.
2. `Errno 2` means the file was not there. A different errno means a different problem, for example a permission you do not have.

In [ ]:
print("the kernel is running in:", os.getcwd())
print("so 'data/orderz.csv' meant that folder, plus data, plus orderz.csv")

## Section 2: reading a CSV by name

Section 1 plus one new element: the file actually opens.

A file format is an agreement about structure. CSV agrees about three things and no more: one row per record, commas between fields, and the first row names the fields. Nothing in that agreement mentions types.

```
the file            what Python receives
4500          ->    "4500"
twelve        ->    "twelve"
(empty cell)  ->    ""
```

In [ ]:
with open(RECORDS_CSV) as f:
    records = list(csv.DictReader(f))

print(f"{len(records)} records")
print(records[0])
print()
print("every value is a", type(records[0]["amount"]).__name__)

`csv.DictReader` takes its keys from the first row of the file. Change the header spelling in the file and every `record["amount"]` in your code raises `KeyError`, which is why the header row is part of the contract and not decoration.

Monday you had one amount stored as text and it broke a comparison. Write those same records to CSV and that bug disappears, because now every amount is text. The defect was never fixed. It was hidden by the format.

## Section 3: convert on purpose, reject with a reason

Section 2 plus one new element: your notebook 1 functions, unchanged, pointed at file data.

These are the same three functions you carved in notebook 1, copied across unchanged. Read them and confirm that nothing in any of them knows it is reading a file.

In [ ]:
def normalise_amount(raw):
    """Convert an amount, or raise ValueError with the interpreter's own wording."""
    return int(raw)


def clean_record(record):
    """Return one record with its amount as a number, or raise ValueError saying what arrived."""
    keeper = dict(record)
    keeper["amount"] = normalise_amount(record["amount"])
    return keeper


def clean_records(rows):
    """Call clean_record on every row and keep the failures, each with its reason."""
    clean = []
    rejects = []
    for r in rows:
        try:
            clean.append(clean_record(r))
        except ValueError as e:
            rejects.append({"id": r["id"], "reason": str(e)})
    return clean, rejects

clean, rejects = clean_records(records)
print(f"input {len(records)}, clean {len(clean)}, rejected {len(rejects)}")
for row in rejects:
    print(row)

### Milestone: where this shows up in production

Public Health England, October 2020, dropped 15,841 COVID cases from reporting. A CSV was converted into an old Excel format that has a hard row limit, and the rows past the limit were silently discarded. Contact tracing never saw those people.

Nobody wrote bad code that day. Somebody did not know the format's contract.

### Interview question this milestone just made answerable

"Everything read from a CSV is a string. What breaks, and where do you convert?"

Answer in three beats: comparisons and arithmetic break first, conversion belongs in one named function, and every failed conversion becomes a rejection with a reason.

## Section 4: one pass, two files out

Section 3 plus one new element: the results leave memory.

Shipping only the clean file is shipping half the job. The rejects file is what lets somebody else fix the source.

In [ ]:
FIELDS = ["id", "segment", "amount", "outcome", "date"]

clean_path = f"{OUTPUT_DIR}/C2_W01_D02_clean_STUDENT.csv"
rejects_path = f"{OUTPUT_DIR}/C2_W01_D02_rejects_STUDENT.csv"

with open(clean_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(clean)

with open(rejects_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "reason"])
    writer.writeheader()
    writer.writerows(rejects)

print("wrote", clean_path)
print("wrote", rejects_path)

`DictWriter` makes you state the field names on the way out. That is you writing the agreement for whoever reads this file next.

The `with` block guarantees the file closes when the block ends, including when your code raises inside it. On your laptop you get away with forgetting. On a server running this a thousand times a day you run out of file handles.

### Writing is not finishing. Reopening is finishing.

In [ ]:
with open(clean_path) as f:
    reopened_clean = list(csv.DictReader(f))
with open(rejects_path) as f:
    reopened_rejects = list(csv.DictReader(f))

print(f"{len(records)} in = {len(reopened_clean)} clean + {len(reopened_rejects)} rejected")
assert len(reopened_clean) + len(reopened_rejects) == len(records), "records went missing"
print("reconciled")
print("first rejection as it reads from disk:", reopened_rejects[0])

## Section 5: the same records, a different agreement

Section 4 plus one new element: a format that agrees about types and nesting.

JSON agrees about more than CSV does. Numbers stay numbers, `null` stays null, and a value can hold another whole record.

In [ ]:
with open(RECORDS_JSON) as f:
    json_records = json.load(f)

print("amount type from JSON:", type(json_records[0]["amount"]).__name__)
print("amount type from CSV: ", type(records[0]["amount"]).__name__)
print()
print(json.dumps(json_records[14], indent=2))

Look at record 1015. Its amount was an empty cell in the CSV, so your cleaning run rejected it. The JSON still carries the original value one level down, inside `source`.

The record was never unrecoverable. The CSV export threw the value away.

### The flattening cost

To put that record into a CSV you have to choose: drop `source`, or invent a column such as `source_amount_raw`. Either way the shape changes, and the person downstream has to be told. That conversation is the cost of flattening.

In [ ]:
missing_amount = [r for r in json_records if r["amount"] is None]

for r in missing_amount:
    raw = r["source"]["amount_raw"]
    try:
        print(f'id {r["id"]}: recovered {normalise_amount(raw)} from the nested block')
    except ValueError:
        print(f'id {r["id"]}: nested block holds {raw!r}, which still will not convert')

### Writing JSON back out

`json.dump` is the mirror of `json.load`. It is worth seeing once, because it is the only way to hand on a record that has a nested block inside it.

In [ ]:
rejects_json_path = f"{OUTPUT_DIR}/C2_W01_D02_rejects_STUDENT.json"

with open(rejects_json_path, "w") as f:
    json.dump(rejects, f, indent=2)

with open(rejects_json_path) as f:
    reopened = json.load(f)

print("wrote and reopened", rejects_json_path)
print(reopened)

`indent=2` is for the human who opens the file next. Without it the whole file is one line, which parses perfectly well and reads terribly.

Notice what you did not have to do: no field names on the way out, because JSON carries the shape with it. That is the same agreement working in your favour for once.

## Section 6: when JSON goes wrong

A JSON file is either wholly valid or wholly unreadable. There is no half-parsed JSON, which is the price of the stronger agreement.

The file below is a vendor feed whose transfer was cut off. Run it and read where the parser says it stopped.

In [ ]:
def load_the_truncated_feed():
    with open(TRUNCATED_JSON) as f:
        return json.load(f)

show_failure(load_the_truncated_feed)

The parser names a line and a column. Go there first, every time.

Your mid-session exercise is this file. Open it, go to the line the error names, and write down what you find. The answer is more interesting than a bad character.

In [ ]:
with open(TRUNCATED_JSON) as f:
    lines = f.read().splitlines()
print("the file has", len(lines), "lines")
print("last two lines:")
for n, line in enumerate(lines[-2:], start=len(lines) - 1):
    print(f"  {n}: {line!r}")

### Interview question this milestone just made answerable

"The vendor's JSON fails at line 47 column 5. What is your first move?"

Open the file at that line. Then decide whether the defect is in the file or in your assumption about the file. Repairing someone else's feed by hand is the last resort, never the first.

## Crux

A file format is an agreement about structure, and everything a CSV agrees to is text. Your job at the boundary is to convert on purpose, reject with a reason, and hand on two files instead of one.

## What tomorrow does with this

Tomorrow hands you the full dataset at its dirtiest, and you point `clean_record` and `clean_records` at it without one edit. The question becomes how many usable records that dataset actually has.

Keep the three functions, not the output files. Tomorrow supplies its own data and calls your code.